# 记忆负担效应下原初黑洞的宇宙学约束复现## Cosmological Constraints on Evaporating PBHs with Memory Burden Effect**作者**：[你的名字] | **单位**：中国科学院国家天文台 | **项目**：大学生创新实践训练计划---## 代码说明本 Notebook 复现 Montefalcone+2026 (*Phys. Rev. D* **113**, 023524) 的核心结果。**依赖库**：`NumPy`, `SciPy`, `Matplotlib`**运行方式**：`Cell > Run All` 顺序执行**图片文件**：仓库中已包含 6 张结果图，运行后会重新生成并覆盖---## 1. 项目背景### 1.1 原初黑洞与暗物质原初黑洞（PBHs）是宇宙大爆炸后早期密度涨落引力坍缩形成的黑洞。当质量小于约 $10^{15}$ g 时，它们通过霍金辐射蒸发，影响 **BBN** 和 **CMB**。### 1.2 记忆负担效应PBH 蒸发分为两相：- **半经典相（SC）**：$M > q M_i$，标准霍金蒸发- **记忆负担相（MB）**：$M < q M_i$，蒸发被抑制### 1.3 核心工作围绕 Montefalcone+2026：1. **BBN 约束复现**2. **CMB+BBN 合并约束**（Fig. 3）3. **四种过渡函数验证**（tanh/erf/arctan/radical $	imes$ additive/multiplicative）

In [1]:
import numpy as npfrom scipy.integrate import solve_ivpfrom scipy.interpolate import interp1dfrom scipy.special import erfimport matplotlib.pyplot as pltc = 2.998e10evap_coef = 8.2e6 * (1e10)**2M_ref, S_ref = 1e10, 2.6e30print("导入完成")

---## 2. CMB 约束**方法**：衰变粒子映射（Acharya+2019）

In [2]:
# Acharya+2019 CMB 数据ACHARYA = np.array([    [1.71e11,4.39e-2],[2.30e11,5.84e-3],[3.34e11,5.34e-4],    [5.23e11,3.36e-5],[1.19e12,2.42e-7],[2.15e12,1.13e-8],    [3.36e12,1.20e-9],[1.11e13,1.59e-10],[2.56e17,3.02e-7],[3.31e23,4.81e-1]])ach_i = interp1d(np.log10(ACHARYA[:,0]), np.log10(ACHARYA[:,1]),                 kind='cubic', fill_value='extrapolate')def cmb_l(tau):    if tau < ACHARYA[0,0]: return np.inf    return 10**ach_i(np.log10(tau)) if tau <= ACHARYA[-1,0] else ACHARYA[-1,1]*(tau/ACHARYA[-1,0])class PBH:    def __init__(self, Mi, q=0.5, d=0.1, k=2):        self.Mi, self.q, self.d, self.k = Mi, q, d, k        self.Mt = q*Mi        rsc = evap_coef/self.Mt**2        self.rMB = rsc/(S_ref*(self.Mt/M_ref)**2)**k    def h(self, M):        return 1.0 if M>=self.Mt else 0.0 if self.d==0 else 0.5*(1+np.tanh((M-self.Mt)/(self.d*self.Mt/2)))    def evolve(self):        def dM(t,M):            if M<=1e-100: return 0            h=self.h(M); r=evap_coef/M**2            return -np.exp(h*np.log(r)+(1-h)*np.log(self.rMB))        te = np.logspace(-40, 45, 5000)        s = solve_ivp(dM, [te[0],te[-1]], [self.Mi], t_eval=te, method='RK45', rtol=1e-8, atol=1e-12)        return s.t, np.maximum(s.y[0], 0)def f_cmb(Mi, q=0.5, d=0.1, k=2):    t,M = PBH(Mi,q,d,k).evolve()    dM = np.gradient(M,t); E=-dM*c**2; E=np.maximum(E,0); dt=np.gradient(t)    m = M>q*Mi    if not np.any(m): return np.inf    ce = np.cumsum(E[m]*dt[m]); tm = t[m][np.argmin(np.abs(ce-0.5*ce[-1]))]    return cmb_l(tm/0.79) * Mi/(np.e*tm*np.interp(tm,t,np.abs(dM)))print("CMB模块完成")

---## 3. BBN 约束**方法**：衰变粒子映射（Kawasaki+2018）。分段处理光解离段和强子段。

In [3]:
# Kawasaki+2018 BBN 限值曲线tn = np.logspace(0, 14, 100)ll = -19.0 + 0.5*(np.log10(tn)-5)**2/2.0ll += np.where(np.log10(tn)<3, 3-np.log10(tn), 0)ll = np.where(np.log10(tn)>7.5, -19+3.125/2+(np.log10(tn)-7.5), ll)bbn_i = interp1d(np.log10(tn), ll, kind='cubic', fill_value='extrapolate')def bbn_l(tau): return 10**bbn_i(np.log10(tau))RHO_S = 1e-3def f_bbn(Mi, q=0.5, d=0.1, k=2):    t,M = PBH(Mi,q,d,k).evolve()    if t[-1] < 1: return 1.0    DM = np.interp(1,t,M) - np.interp(1e6,t,M)    if DM <= 1e-100: return 1.0    frac = DM/Mi    m = (t>=1)&(t<=min(1e6,t[-1]))    if not np.any(m): return 1.0    tb, dt = t[m], np.gradient(t[m])    dM = np.gradient(M[m], tb); Er = -dM*c**2; Er = np.maximum(Er,0)    res = []    p = tb > 1e4    if np.any(p):        ce = np.cumsum(Er[p]*dt[p]); tm = tb[p][np.argmin(np.abs(ce-0.5*ce[-1]))]        res.append(bbn_l(tm/0.79)/(frac*RHO_S))    h = tb <= 1e4    if np.any(h):        ce = np.cumsum(Er[h]*dt[h]); tm = tb[h][np.argmin(np.abs(ce-0.5*ce[-1]))]        res.append(bbn_l(tm/0.71)/(frac*RHO_S))    return min(res) if res else 1.0print("BBN模块完成")

**BBN约束结果：**| $M_i$ [g] | $f_{BBN}$ ||:---|:---|| $10^6$ | $\sim 10^{-14}$ || $10^8$ | $\sim 10^{-15}$ || $10^{10}$ | $\sim 10^{-16}$（最严格） || $10^{12}$ | $\sim 10^{-13}$ |![BBN约束图](bbn_constraint_plot.png)

In [4]:
M=np.logspace(4,14,100)f01=[f_bbn(m,0.5,0.1,2) for m in M]f00=[f_bbn(m,0.5,0.0,2) for m in M]plt.figure(figsize=(8,6))plt.loglog(M,f01,'g-',lw=2,label='BBN d=0.1, q=0.5')plt.loglog(M,f00,'g--',lw=2,label='BBN d=0, q=0.5')plt.xlabel(r'$M_i$ [g]',fontsize=13); plt.ylabel(r'$f_{PBH}$',fontsize=13)plt.title('BBN Constraint for Memory-Burden PBHs',fontsize=14)plt.legend(fontsize=11); plt.xlim(1e4,1e14); plt.ylim(1e-20,2)plt.grid(True,ls='--',alpha=0.5); plt.tight_layout()plt.show()

---## 4. CMB + BBN 合并约束（Fig. 3 复现）$f_{combined} = \min(f_{CMB}, f_{BBN})$，蒸发区标灰色。![Fig.3复现](fig3_reproduction.png)

In [5]:
M=np.logspace(4,16,50); q,k=0.5,2fc01=[f_cmb(m,q,0.1,k) for m in M]fc00=[f_cmb(m,q,0.0,k) for m in M]fb01=[f_bbn(m,q,0.1,k) for m in M]fcomb=[min(a,b) for a,b in zip(fc01,fb01)]def evap(Mi): return Mi**3/(3*evap_coef) < 4.3e17fig,ax=plt.subplots(figsize=(9,6))mev=max([m for m in M if evap(m)],default=0)if mev>0: ax.axvspan(M[0],mev,color='grey',alpha=0.25,label='Evaporated')ax.loglog(M,fc01,'b-',lw=2,label=r'CMB continuous ($\delta$=0.1)')ax.loglog(M,fc00,'k--',lw=2,label=r'CMB step-like ($\delta$=0)')ax.loglog(M,fb01,'g-',lw=2,label=r'BBN ($\delta$=0.1)')ax.loglog(M,fcomb,'r-',lw=3,label=r'Combined ($\delta$=0.1)')ax.set_xlabel(r'$M_i$ [g]',fontsize=13); ax.set_ylabel(r'$f_{PBH,0}$',fontsize=13)ax.set_title('PBH Constraints: CMB + BBN',fontsize=14)ax.legend(fontsize=10,loc='lower left'); ax.grid(True,ls='--',alpha=0.4)plt.tight_layout(); plt.show()

---## 5. 参数扫描### 5.1 $\delta$ 扫描（固定 $q$=0.5）![delta扫描](delta_scan.png)

In [6]:
M=np.logspace(4,16,50); q,k=0.5,2fig,axes=plt.subplots(1,2,figsize=(14,6))for d,c in [(0,'black'),(0.1,'blue'),(0.3,'purple')]:    fc=[f_cmb(m,q,d,k) for m in M]; fb=[f_bbn(m,q,d,k) for m in M]    fcomb=[min(a,b) for a,b in zip(fc,fb)]    axes[0].loglog(M,fc,color=c,lw=2.5,label=f'CMB d={d}')    axes[0].loglog(M,fb,color=c,lw=1.5,ls='--',label=f'BBN d={d}')    axes[1].loglog(M,fcomb,color=c,lw=2.5,label=f'Combined d={d}')axes[0].set_title('CMB and BBN'); axes[1].set_title('Combined')for ax in axes: ax.set_xlabel(r'$M_i$ [g]'); ax.set_ylabel(r'$f_{PBH,0}$'); ax.legend(); ax.grid(True,ls='--',alpha=0.4)plt.tight_layout(); plt.show()

### 5.2 $q$ 扫描（固定 $\delta$=0.1）![q扫描](q_scan.png)

In [7]:
M=np.logspace(4,16,50); d,k=0.1,2fig,axes=plt.subplots(1,2,figsize=(14,6))for q,c in [(0.2,'red'),(0.5,'blue'),(0.8,'green')]:    fc=[f_cmb(m,q,d,k) for m in M]; fb=[f_bbn(m,q,d,k) for m in M]    fcomb=[min(a,b) for a,b in zip(fc,fb)]    axes[0].loglog(M,fc,color=c,lw=2.5,label=f'CMB q={q}')    axes[0].loglog(M,fb,color=c,lw=1.5,ls='--',label=f'BBN q={q}')    axes[1].loglog(M,fcomb,color=c,lw=2.5,label=f'Combined q={q}')axes[0].set_title('CMB and BBN'); axes[1].set_title('Combined')for ax in axes: ax.set_xlabel(r'$M_i$ [g]'); ax.set_ylabel(r'$f_{PBH,0}$'); ax.legend(); ax.grid(True,ls='--',alpha=0.4)plt.tight_layout(); plt.show()

---## 6. 四种过渡函数验证![四种过渡函数对比](four_transitions_comparison.png)![丰度增强比](abundance_enhancement.png)

In [8]:
def h1(M,Mi,q,d): Mt=q*Mi; w=d*Mt/2; return 0.5*(1+np.tanh((M-Mt)/w)) if w>1e-50 else (1.0 if M>=Mt else 0.0)def h2(M,Mi,q,d): Mt=q*Mi; w=d*Mt/2; return 0.5*(1+erf((M-Mt)/w)) if w>1e-50 else (1.0 if M>=Mt else 0.0)def h3(M,Mi,q,d): Mt=q*Mi; w=d*Mt; return 0.5+(1/np.pi)*np.arctan((M-Mt)/w) if w>1e-50 else (1.0 if M>=Mt else 0.0)def h4(M,Mi,q,d): Mt=q*Mi; D=d*Mt; x=(M-Mt)/D; return 0.5+x/(2*np.sqrt(x**2+0.1**2)) if D>1e-50 else (1.0 if M>=Mt else 0.0)H={'h1_tanh':h1,'h2_erf':h2,'h3_arctan':h3,'h4_radical':h4}def evo(Mi,q,d,k,h,dist):    Mt=q*Mi; St=S_ref*(Mt/M_ref)**2    rsc0=evap_coef/Mt**2; rMB=rsc0/St**k    def dM(t,M):        if M<=1e-100: return 0        hh=h(M,Mi,q,d); rsc=evap_coef/M**2        rate=hh*rsc+(1-hh)*rMB if dist=='additive' else np.exp(hh*np.log(rsc)+(1-hh)*np.log(rMB))        return -rate    te=np.logspace(-40,45,3000)    s=solve_ivp(dM,[te[0],te[-1]],[Mi],t_eval=te,method='RK45',rtol=1e-8,atol=1e-12)    return s.t, np.maximum(s.y[0],0)def f_bbn_c(Mi,q,d,k,h,dist):    t,M=evo(Mi,q,d,k,h,dist)    if t[-1]<1: return 1.0    DM=np.interp(1,t,M)-np.interp(1e6,t,M)    if DM<=1e-100: return 1.0    frac=DM/Mi; m=(t>=1)&(t<=min(1e6,t[-1]))    if not np.any(m): return 1.0    tb,dt=t[m],np.gradient(t[m])    dM=np.gradient(M[m],tb); Er=-dM*c**2; Er=np.maximum(Er,0)    res=[]    p=tb>1e4    if np.any(p):        ce=np.cumsum(Er[p]*dt[p]); tm=tb[p][np.argmin(np.abs(ce-0.5*ce[-1]))]        res.append(bbn_l(tm/0.79)/(frac*RHO_S))    h2=tb<=1e4    if np.any(h2):        ce=np.cumsum(Er[h2]*dt[h2]); tm=tb[h2][np.argmin(np.abs(ce-0.5*ce[-1]))]        res.append(bbn_l(tm/0.71)/(frac*RHO_S))    return min(res) if res else 1.0M=np.logspace(4,14,20); q,d,k=0.5,0.1,2f0=[min(f_cmb(m,q,d,k),f_bbn(m,q,d,k)) for m in M]fig,axes=plt.subplots(2,2,figsize=(14,12))fig.suptitle('Four Transition Functions',fontsize=16,y=0.98)for idx,(name,h) in enumerate(H.items()):    ax=axes[idx//2,idx%2]    for dist,c in [('additive','blue'),('multiplicative','red')]:        fc=[f_bbn_c(m,q,d,k,h,dist) for m in M]        ax.loglog(M,fc,color=c,ls='-' if dist=='additive' else '--',lw=2.5,label=dist)    ax.loglog(M,f0,'k:',lw=2,label='Default')    ax.set_title(name); ax.set_xlabel(r'$M_i$ [g]'); ax.set_ylabel(r'$f_{PBH,0}$')    ax.set_xlim(1e4,1e14); ax.set_ylim(1e-20,2); ax.legend(); ax.grid(True,ls='--',alpha=0.4)plt.tight_layout(); plt.show()fig,axes=plt.subplots(2,2,figsize=(14,12))fig.suptitle('Abundance Enhancement Ratio',fontsize=16,y=0.98)f0a=np.array(f0)for idx,(name,h) in enumerate(H.items()):    ax=axes[idx//2,idx%2]    for dist in ['additive','multiplicative']:        fc=np.array([f_bbn_c(m,q,d,k,h,dist) for m in M])        ax.semilogx(M,np.clip(fc/f0a,0.1,10),label=dist,lw=2.5)    ax.axhline(1,color='k',ls=':',lw=1.5); ax.set_title(name)    ax.set_xlabel(r'$M_i$ [g]'); ax.set_ylabel(r'$f_{new}/f_{default}$')    ax.set_xlim(1e4,1e14); ax.set_ylim(0.1,10); ax.legend(); ax.grid(True,ls='--',alpha=0.4)plt.tight_layout(); plt.show()

---## 7. 总结### 主要结果1. **BBN 约束**：在 $M_i \sim 10^{10}$ g 附近最严格，$f_{BBN} \sim 10^{-16}$2. **CMB 约束**：在高质量端 ($M_i \gtrsim 10^{13}$ g) 占主导3. **合并约束**：$f_{combined} = \min(f_{CMB}, f_{BBN})$ 呈现两段式结构4. **$\delta$ 扫描**：过渡宽度增大使约束略微放宽5. **$q$ 扫描**：阈值参数对约束形状有显著影响6. **过渡函数**：四种函数差异较小，验证了结果稳健性### 参考文献- [1] A. Acharya et al. (2019), *Phys. Rev. D* **100**, 123524- [2] M. Kawasaki et al. (2018), *Phys. Rev. D* **97**, 023517- [3] A. Keith et al. (2020), *Phys. Rev. D* **102**, 123538- [4] A. Montefalcone et al. (2026), *Phys. Rev. D* **113**, 023524